# 安裝套件


In [ ]:
!pip install datasets==2.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
!pip show datasets

Name: datasets
Version: 2.21.0
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dill, filelock, fsspec, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: evaluate, torchtune


# BERT-base-uncased 多輸出基線模型


In [ ]:
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from evaluate import load
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.notebook import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Normalize punctuation to reduce [UNK] tokens in BERT tokenization.
# 將常見中文標點正規化為英文標點，避免 BERT（以英文詞彙表為主）把中文標點切成 [UNK]，
# 以降低噪音、提升句對表示的穩定性（對相關度回歸與蕴含分類皆有幫助）。
token_replacement = [
    ["：" , ":"],
    ["，" , ","],
    ["“" , "\""],
    ["”" , "\""],
    ["？" , "?"],
    ["……" , "..."],
    ["！" , "!"]
]

In [ ]:
# 從 Hugging Face 載入 "google-bert/bert-base-uncased" 預訓練模型的tokenizer
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
# PyTorch Dataset：SemEval-2014 Task 1（句子配對 Sentence Pair）
# - 每筆樣本輸入為兩句話（句子A/句子B；資料欄位名為 premise/hypothesis）
#   (1) relatedness_score：語意相關度（1~5，回歸）
#   (2) entailment_judgement：文本蘊含三分類（Neutral / Entailment / Contradiction）
class SemevalDataset(Dataset):
    # 初始化：split 指定要載入的資料切分（train / validation / test）
    def __init__(self, split="train") -> None:
        super().__init__()
        # 確保 split 參數合法，避免誤用錯誤的資料切分
        assert split in ["train", "validation", "test"]

        # 使用 Hugging Face Datasets 載入資料集
        # - trust_remote_code=True：此資料集需要執行其 dataset script 才能正確載入
        # - cache_dir：將下載/處理後的資料快取在本地，方便重跑不同實驗
        # - to_list()：轉成 Python list，便於在 __getitem__ 以 index 取資料
        self.data = load_dataset(
            "sem_eval_2014_task_1",
            split=split,
            trust_remote_code=True,
            cache_dir="./cache/",
        ).to_list()

    # 依 index 取出一筆資料（句子配對 + 兩個 label）
    def __getitem__(self, index):
        d = self.data[index]

        # 文字正規化：將常見中文標點替換為英文標點，降低英文詞彙表 tokenizer 產生 [UNK]/稀有符號的機率，tokenization 更穩定。
        for k in ["premise", "hypothesis"]:
            for src, tgt in token_replacement:
                d[k] = d[k].replace(src, tgt)

        return d

    # 回傳此 split 的樣本數
    def __len__(self):
        return len(self.data)

# 剪查
data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Generating train split:   0%|          | 0/4500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4927 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [ ]:
# 超參數
lr = 2e-5
epochs = 20
train_batch_size = 16
validation_batch_size = 16

In [ ]:
def collate_fn(batch):
    """
    1) 把「句子A/句子B」的文字批次化後交給 tokenizer 一次處理（效率較好）
    2) 同時整理兩個任務的標籤：relatedness_score（回歸）與 entailment_judgment（三分類）
    DataLoader 每次取 batch 時都會呼叫 collate_fn 來把原始樣本打包成模型可吃的 tensor
    """

    # 從 batch 抽出句子配對（欄位 premise/hypothesis）
    premises = [d['premise'] for d in batch]
    hypotheses = [d['hypothesis'] for d in batch]

    # 用 tokenizer 將句對轉成模型輸入（BERT 會自動組成 [CLS] A [SEP] B [SEP]）
    # BERT 會自動處理成 [CLS] premise [SEP] hypothesis [SEP]
    inputs = tokenizer(
        premises,
        hypotheses,
        padding=True,  # 同一批次中的序列填充到相同的長度 
        truncation=True,  # 超過最大長度的 截斷
        return_tensors="pt" # 回傳 PyTorch (pt) Tensors
    )

    # 提取兩個子任務的labels
    # 回歸: relatedness_score
    labels1 = [d['relatedness_score'] for d in batch]
    # 分類: entailment_judgment
    labels2 = [d['entailment_judgment'] for d in batch]

    # 將標籤轉換為 PyTorch Tensors
    # 回歸任：FloatTensor
    labels1_tensor = torch.tensor(labels1, dtype=torch.float)
    # 分類任務：LongTensor (計算 CrossEntropyLoss)
    labels2_tensor = torch.tensor(labels2, dtype=torch.long)

    # 5. 回傳一個字典，包含模型輸入和兩個任務的標籤
    return {
        "input_txt": inputs,       # 字典，包含 'input_ids', 'token_type_ids', 'attention_mask'
        "labels1": labels1_tensor, # 回歸任務的標籤
        "labels2": labels2_tensor  # 分類任務的標籤
    }

# 我把資料分成 train/validation/test 三個 split，並各自建立 DataLoader
ds_train = SemevalDataset(split="train")
ds_validation = SemevalDataset(split="validation")
ds_test = SemevalDataset(split="test")



dl_train = DataLoader(
    ds_train,
    batch_size=train_batch_size,
    shuffle=True,  # 打亂 
    collate_fn=collate_fn  
    )

dl_validation = DataLoader(
    ds_validation,
    batch_size=validation_batch_size,
    shuffle=False, 不需要打亂
    collate_fn=collate_fn)

dl_test = DataLoader(
    ds_test,
    batch_size=1, 
    shuffle=False,
    collate_fn=collate_fn)


In [ ]:
class MultiLabelModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 載入 "google-bert/bert-base-uncased" 預訓練模型：只回傳最後一層的隱藏狀態
        self.bert = BertModel.from_pretrained(
            "google-bert/bert-base-uncased",
            cache_dir="./cache/"
        )

        # 取得 BERT 模型的輸出維度 (bert-base 為 768)
        bert_hidden_size = self.bert.config.hidden_size

        # 定義子回歸任務
        # 輸入: 768、輸出維度: 1
        self.regressor = torch.nn.Linear(bert_hidden_size, 1)

        # 定義分類任務
        # 輸入維度: 768、輸出維度: 3 ('NEUTRAL', 'ENTAILMENT', 'CONTRADICTION')
        self.classifier = torch.nn.Linear(bert_hidden_size, 3)

    def forward(self, **kwargs):
        # 將 input_txt 傳給 BERT 模型
        # input_txt 會自動unpack字典中的 'input_ids', 'token_type_ids', 'attention_mask'
        outputs = self.bert(**input_txt)

        # 取得 BERT 的 pooler_output
        # pooler_output 是 [CLS] token 經過一個線性層和 Tanh 激活函數後的特徵，常用於句子層級的任務
        # 維度:(batch_size, hidden_size)
        pooler_output = outputs.pooler_output

        # 將 pooler_output 分別傳遞給兩個任務的頭

        # 回歸
        # 輸出維度: (batch_size, 1)
        output1 = self.regressor(pooler_output)

        # 方便計算loss，將 (batch_size, 1)壓縮為(batch_size)
        output1 = output1.squeeze(-1)

        # 分類任務
        # 輸出維度: (batch_size, 3)
        output2 = self.classifier(pooler_output)

        # 回傳輸出
        return output1, output2

In [ ]:
model = MultiLabelModel().to(device)
# 使用 AdamW 優化器，傳入模型所有可訓練的參數 (model.parameters()) 和學習率 (lr)
optimizer = AdamW(model.parameters(), lr=lr,weight_decay=0.01)

# 回歸任務的損失函數：Mean Squared Error Loss
loss_fn_1 = torch.nn.MSELoss()

# 分類的損失函數：Cross-Entropy Loss(會自動處理 logits(模型的原始輸出)和整數標籤)
loss_fn_2 = torch.nn.CrossEntropyLoss()

# scoring functions
# 載入 Hugging Face evaluate 的評分函式
psr = load("pearsonr")
acc = load("accuracy")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
# 用來儲存最佳分數
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train() # 設定訓練模式

    total_train_loss = 0.0 # 計算平均訓練損失

    for batch in pbar:
        # 1. 資料移至device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 2. 清除梯度
        optimizer.zero_grad()

        # 3. 前向傳播 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 4. 計算損失
        loss1 = loss_fn_1(outputs1, labels1) # 回歸任務 (MSE)
        loss2 = loss_fn_2(outputs2, labels2) # 分類任務 (CrossEntropy)

        # 總損失
        loss = loss1 + loss2

        # 5. 反向傳播
        loss.backward()

        # 6. 模型優化
        optimizer.step()

        # 更新顯示
        pbar.set_postfix(loss=loss.item())
        total_train_loss += loss.item()

    print(f"Epoch {ep+1} Average Train Loss: {total_train_loss / len(dl_train):.4f}")

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval() # 設定為評估模式
    # TODO5: Write the evaluation loop
    # Write your code here
    # Evaluate your model
    with torch.no_grad():
        for batch in pbar:
            # 1. 將資料移至 device
            input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
            labels1 = batch['labels1'].to(device)
            labels2 = batch['labels2'].to(device)

            # 2. 執行模型 
            outputs1, outputs2 = model(input_txt=input_txt)

            # 3. 準備評分
            # outputs2 是 (batch_size, 3) 的 logits，我們需要 argmax 取得預測的類別 (0, 1, or 2)
            predictions2 = torch.argmax(outputs2, dim=1)

            # 4. 將這一批次的結果加入評分器
            psr.add_batch(predictions=outputs1, references=labels1)
            acc.add_batch(predictions=predictions2, references=labels2)

    # 5. 迴圈結束，計算總體分數
    pearson_corr = psr.compute()['pearsonr']
    accuracy = acc.compute()['accuracy']
    # print(f"F1 Score: {f1.compute()}")
    # print(f"Epoch {ep+1} Validation Pearson: {pearson_corr:.4f}, Accuracy: {accuracy:.4f}")
    if pearson_corr + accuracy > best_score:
        best = pearson_corr + accuracy
        torch.save(model.state_dict(), f'/content/drive/MyDrive/saved_models/best_model_4.ckpt')


  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 1 Average Train Loss: 1.5777


  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 20 Average Train Loss: 0.0580


  0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
# 測試迴圈

# 載入模型權重
print("Loading best model from best_model_4.ckpt")
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"best_model_4.ckpt", weights_only=True))

# 設定 pbar 和 model.eval()
pbar = tqdm(dl_test, desc="Test")
model.eval() # 將模型設定為評估模式

# 執行測試迴圈
print("Starting evaluation on test set...")
with torch.no_grad():
    for batch in pbar:
        # 將資料移至 device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 執行模型 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 評分
        # outputs2 是 (batch_size, 3) 的 logits，我們需要 argmax 取得預測的類別
        predictions2 = torch.argmax(outputs2, dim=1)

        # 結果加入評分器
        psr.add_batch(predictions=outputs1, references=labels1)
        acc.add_batch(predictions=predictions2, references=labels2)

# 迴圈結束，計算最終的測試集分數
test_pearson_corr = psr.compute()['pearsonr']
test_accuracy = acc.compute()['accuracy']

# 結果
print("--- Test Set Results ---")
print(f"Test Pearson Correlation: {test_pearson_corr:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("--------------------------")

Loading best model from /content/drive/MyDrive/saved_models/best_model_4.ckpt


Test:   0%|          | 0/4927 [00:00<?, ?it/s]

Starting evaluation on test set...
--- Test Set Results ---
Test Pearson Correlation: 0.8900
Test Accuracy: 0.8910
--------------------------


# 訓練兩個新的 BERT-base 模型：

模型 A： 只在「子任務1」（relatedness_score 回歸）上訓練。



In [ ]:
# PyTorch Dataset類別，用來處理SemEval 2014 Task 1資料集
class SemevalDataset(Dataset):
    #參數 split 用來指定要載入 'train', 'validation', 或是 'test' 資料集
    def __init__(self, split="train") -> None:
        super().__init__()
        # 確保傳入的 split 參數是合法的
        assert split in ["train", "validation", "test"]
        # 使用 Hugging Face 的 datasets 函式庫載入 "sem_eval_2014_task_1" 資料集
        # trust_remote_code=True 是因為這個資料集需要執行遠端程式碼來載入
        # .to_list() 載入的資料轉為Python 列表
        self.data = load_dataset(
            "sem_eval_2014_task_1", split=split, trust_remote_code=True, cache_dir="./cache/"
        ).to_list()

    # 定義如何根據index取得單筆資料
    def __getitem__(self, index):
        # 根據索引從 self.data 列表中取出原始資料
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        # premise hypothesis欄位執行標點符號的替換
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d
    # 回傳資料筆數
    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [ ]:
def collate_fn(batch):

    # 將 'premise' 和 'hypothesis' 欄位分別抽出來，放到兩個列表
    premises = [d['premise'] for d in batch]
    hypotheses = [d['hypothesis'] for d in batch]

    # 使用 tokenizer 將文本對轉換為模型輸入格式
    inputs = tokenizer(
        premises,
        hypotheses,
        padding=True,  # 填充
        truncation=True,  # 截斷
        return_tensors="pt" # 回傳 PyTorch (pt) Tensors
    )

    # 提取兩個子任務的labels
    # 回歸: relatedness_score
    labels1 = [d['relatedness_score'] for d in batch]
    # 分類: entailment_judgment
    labels2 = [d['entailment_judgment'] for d in batch]

    # 將標籤轉換為 PyTorch Tensors
    # 回歸任 FloatTensor
    labels1_tensor = torch.tensor(labels1, dtype=torch.float)
    # 分類任務LongTensor (計算CrossEntropyLoss)
    labels2_tensor = torch.tensor(labels2, dtype=torch.long)

    # 回傳字典，包含模型輸入和兩個任務的標籤
    return {
        "input_txt": inputs,       # 字典，包含 'input_ids', 'token_type_ids', 'attention_mask'
        "labels1": labels1_tensor, # 回歸任務的標籤
        "labels2": labels2_tensor  # 分類任務的標籤
    }

# 訓練、驗證和測試資料集
ds_train = SemevalDataset(split="train")
ds_validation = SemevalDataset(split="validation")
ds_test = SemevalDataset(split="test")


# DataLoader
dl_train = DataLoader(
    ds_train,
    batch_size=train_batch_size,
    shuffle=True,  
    collate_fn=collate_fn  
    )

dl_validation = DataLoader(
    ds_validation,
    batch_size=validation_batch_size,
    shuffle=False, 
    collate_fn=collate_fn)

dl_test = DataLoader(
    ds_test,
    batch_size=1, 
    shuffle=False,
    collate_fn=collate_fn)


In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

# --- 定義 "只做回歸" 的模型 A ---
class RegressionModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # 載入 BERT-base
        self.bert = BertModel.from_pretrained(
            "google-bert/bert-base-uncased",
            cache_dir="./cache/"
        )
        bert_hidden_size = self.bert.config.hidden_size

        # 防止過擬合
        self.dropout = torch.nn.Dropout(0.1)

        # --- 只有 Regressor ---
        self.regressor = torch.nn.Linear(bert_hidden_size, 1)

    def forward(self, input_txt):
        # 執行 BERT
        outputs = self.bert(**input_txt)
        pooler_output = outputs.pooler_output
        dropped_output = self.dropout(pooler_output)

        # --- 只回傳任務1的輸出 ---
        output1 = self.regressor(dropped_output).squeeze(-1)
        return output1

print("RegressionModel (Model A) class defined.")

RegressionModel (Model A) class defined.


In [ ]:
# --- 實驗 A: 只訓練回歸 (任務1) ---
print("--- Starting Experiment A: Regression ONLY ---")

# 超參數
lr = 2e-5
epochs = 20 # (例如，使用 10)
train_batch_size = 16
validation_batch_size = 16

# 確保資料夾存在
os.makedirs('./saved_models', exist_ok=True)

# 載入模型、優化器、損失函數
model_A = RegressionModel().to(device)
optimizer_A = AdamW(model_A.parameters(), lr=lr, weight_decay=0.01) # (使用您最佳的優化器設定)
loss_fn_A = nn.MSELoss() # <--- 只使用 MSE Loss

psr = load("pearsonr")


# 訓練和驗證迴圈
best_score_A = -1.0 # (Pearsonr 可能是負數，故從 -1 開始)

for ep in range(epochs):

    # Model A訓練
    model_A.train()
    pbar = tqdm(dl_train, desc=f"[Model A] Reg Epoch {ep+1}/{epochs}")
    for batch in pbar:
        # (DataLoaders 保持不變，照常載入所有資料)
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)

        optimizer_A.zero_grad()

        # 只執行模型 A 
        outputs1 = model_A(input_txt=input_txt)

        # 只計算 loss 1 
        loss = loss_fn_A(outputs1, labels1)

        loss.backward()
        optimizer_A.step()

        pbar.set_postfix(loss=loss.item())

    # Model A驗證
    model_A.eval()

    # 重置 metric
    psr = load("pearsonr")

    with torch.no_grad():
        for batch in tqdm(dl_validation, desc="[Model A] Reg Validation"):
            input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
            labels1 = batch['labels1'].to(device)

            outputs1 = model_A(input_txt=input_txt)

            # --- 只評估 pearsonr ---
            psr.add_batch(predictions=outputs1, references=labels1)

    pearson_corr = psr.compute()['pearsonr']
    print(f"Epoch {ep+1} Validation Pearson: {pearson_corr:.4f}")

    # 根據 pearson_corr 儲存模型
    if pearson_corr > best_score_A:
        best_score_A = pearson_corr
        print(f"*** New best Pearson score: {best_score_A:.4f}. Saving model... ***")
        torch.save(model_A.state_dict(), f'/content/drive/MyDrive/saved_models/best_model_REGRESSION_ONLY.ckpt')

--- Starting Experiment A: Regression ONLY ---


[Model A] Reg Epoch 1/20:   0%|          | 0/282 [00:00<?, ?it/s]

[Model A] Reg Validation:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1 Validation Pearson: 0.8485
*** New best Pearson score: 0.8485. Saving model... ***


[Model A] Reg Epoch 2/20:   0%|          | 0/282 [00:00<?, ?it/s]

...

[Model A] Reg Validation:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 19 Validation Pearson: 0.8854


[Model A] Reg Epoch 20/20:   0%|          | 0/282 [00:00<?, ?it/s]

[Model A] Reg Validation:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 20 Validation Pearson: 0.8873


In [ ]:
# --- Model A測試 ---
print("--- Testing Regression ONLY model (Model A) ---")

# 載入模型 A 的權重
model_A = RegressionModel().to(device)
model_A.load_state_dict(torch.load(f'best_model_REGRESSION_ONLY.ckpt'))
model_A.eval()

#重新載入乾淨的 metric
psr = load("pearsonr")

# 執行測試迴圈
with torch.no_grad():
    for batch in tqdm(dl_test, desc="[Model A] Reg Test"):
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)

        outputs1 = model_A(input_txt=input_txt)

        psr.add_batch(predictions=outputs1, references=labels1)

# 分數
test_pearson = psr.compute()['pearsonr']
print(f"--- FINAL Test Pearson (Regression ONLY): {test_pearson:.4f} ---")

--- Testing Regression ONLY model (Model A) ---


[Model A] Reg Test:   0%|          | 0/4927 [00:00<?, ?it/s]

--- FINAL Test Pearson (Regression ONLY): 0.8945 ---


模型 B： 只在「子任務2」（entailment_judgment 分類）上訓練。

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

# --- 定義 "只做分類" 的模型 B ---
class ClassificationModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # 載入 BERT-base
        self.bert = BertModel.from_pretrained(
            "google-bert/bert-base-uncased",
            cache_dir="./cache/"
        )
        bert_hidden_size = self.bert.config.hidden_size

        # 留 Dropout 
        self.dropout = torch.nn.Dropout(0.1)

        # --- 只有子任務2的頭 (Classifier) ---
        self.classifier = torch.nn.Linear(bert_hidden_size, 3)

    def forward(self, input_txt):
        # 執行 BERT
        outputs = self.bert(**input_txt)
        pooler_output = outputs.pooler_output
        dropped_output = self.dropout(pooler_output)

        # --- 只回傳任務2的輸出 ---
        output2 = self.classifier(dropped_output)
        return output2

print("ClassificationModel (Model B) class defined.")

ClassificationModel (Model B) class defined.


In [ ]:
from torch.optim import AdamW # <--- 修正：AdamW 來自 torch.optim
from evaluate import load
from tqdm.notebook import tqdm
import os
import torch.nn as nn

# --- 只訓練分類 (任務2) ---
print("--- Starting Experiment B: Classification ONLY ---")

lr = 2e-5
epochs = 20 # (例如，使用 10)
train_batch_size = 16
validation_batch_size = 16

# 確保資料夾在
os.makedirs('./saved_models', exist_ok=True)

# 載入模型、優化器、損失函數
model_B = ClassificationModel().to(device)
optimizer_B = AdamW(model_B.parameters(), lr=lr, weight_decay=0.01)
loss_fn_B = nn.CrossEntropyLoss() # <--- 只使用 CrossEntropy Loss

# 重新載入評分函式
acc = load("accuracy")

# 訓練和驗證迴圈
best_score_B = 0.0 

for ep in range(epochs):

    # Model B
    model_B.train()
    pbar = tqdm(dl_train, desc=f"[Model B] Cls Epoch {ep+1}/{epochs}")
    for batch in pbar:
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        # 取得 labels2 
        labels2 = batch['labels2'].to(device)

        optimizer_B.zero_grad()

        # 只執行模型 B
        outputs2 = model_B(input_txt=input_txt)

        # 計算 loss2 
        loss = loss_fn_B(outputs2, labels2)

        loss.backward()
        optimizer_B.step()

        pbar.set_postfix(loss=loss.item())

    # 驗證
    model_B.eval()

    acc = load("accuracy")

    with torch.no_grad():
        for batch in tqdm(dl_validation, desc="[Model B] Cls Validation"):
            input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
            labels2 = batch['labels2'].to(device)

            outputs2 = model_B(input_txt=input_txt)

            #轉換 logits 並評估 accuracy
            predictions2 = torch.argmax(outputs2, dim=1)
            acc.add_batch(predictions=predictions2, references=labels2)

    accuracy = acc.compute()['accuracy']
    print(f"Epoch {ep+1} Validation Accuracy: {accuracy:.4f}")

    # 根據 accuracy 儲存模型 
    if accuracy > best_score_B:
        best_score_B = accuracy
        print(f"*** New best Accuracy score: {best_score_B:.4f}. Saving model... ***")
        torch.save(model_B.state_dict(), f'/content/drive/MyDrive/saved_models/best_model_CLASSIFICATION_ONLY.ckpt')

--- Starting Experiment B: Classification ONLY ---


[Model B] Cls Epoch 1/20:   0%|          | 0/282 [00:00<?, ?it/s]

[Model B] Cls Validation:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1 Validation Accuracy: 0.8000
*** New best Accuracy score: 0.8000. Saving model... ***


[Model B] Cls Epoch 2/20:   0%|          | 0/282 [00:00<?, ?it/s]

...

[Model B] Cls Validation:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 19 Validation Accuracy: 0.8400


[Model B] Cls Epoch 20/20:   0%|          | 0/282 [00:00<?, ?it/s]

[Model B] Cls Validation:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 20 Validation Accuracy: 0.8400


In [ ]:
# 測試 
print("--- Testing Classification ONLY model (Model B) ---")

# 載入模型 B 的權重
model_B = ClassificationModel().to(device)
model_B.load_state_dict(torch.load(f'best_model_CLASSIFICATION_ONLY.ckpt'))
model_B.eval()

# 載入 metric
acc = load("accuracy")

# 測試迴圈
with torch.no_grad():
    for batch in tqdm(dl_test, desc="[Model B] Cls Test"):
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels2 = batch['labels2'].to(device)

        outputs2 = model_B(input_txt=input_txt)

        predictions2 = torch.argmax(outputs2, dim=1)
        acc.add_batch(predictions=predictions2, references=labels2)

# 分數
test_accuracy = acc.compute()['accuracy']
print(f"--- FINAL Test Accuracy (Classification ONLY): {test_accuracy:.4f} ---")

--- Testing Classification ONLY model (Model B) ---


[Model B] Cls Test:   0%|          | 0/4927 [00:00<?, ?it/s]

--- FINAL Test Accuracy (Classification ONLY): 0.8567 ---


#  RoBERTa-base 模型

In [ ]:
# PyTorch Dataset類別，用來處理SemEval 2014 Task 1資料集
class SemevalDataset(Dataset):
    # 初始化函式，參數 split 用來指定要載入 'train', 'validation', 或是 'test' 資料集
    def __init__(self, split="train") -> None:
        super().__init__()
        # 斷言 (assert) 確保傳入的 split 參數是合法的
        assert split in ["train", "validation", "test"]
        # 使用 Hugging Face 的 datasets 函式庫載入 "sem_eval_2014_task_1" 資料集
        # trust_remote_code=True 是因為這個資料集需要執行遠端程式碼來載入
        # .to_list() 載入的資料轉為Python 列表
        self.data = load_dataset(
            "sem_eval_2014_task_1", split=split, trust_remote_code=True, cache_dir="./cache/"
        ).to_list()

    # 定義如何根據索引 (index) 取得單筆資料
    def __getitem__(self, index):
        # 根據索引從 self.data 列表中取出原始資料
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        # 針對 "premise" (前提) 和 "hypothesis" (假設) 這兩個欄位，執行標點符號的替換
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d
    # 回傳資料集的總資料筆數
    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [ ]:
#超參數
lr = 2e-5
epochs = 20
train_batch_size = 16
validation_batch_size = 16

In [ ]:
def collate_fn(batch):

    # 將 'premise' 和 'hypothesis' 欄位分別抽出來，放到兩個列表
    premises = [d['premise'] for d in batch]
    hypotheses = [d['hypothesis'] for d in batch]

    # 使用 tokenizer 將文本對(premise,hypothesis)轉換為模型輸入格式
    # BERT 會自動處理成 [CLS] premise [SEP] hypothesis [SEP]
    inputs = tokenizer(
        premises,
        hypotheses,
        padding=True,  # 充到
        truncation=True,  # 截斷
        return_tensors="pt" 
    )

    # 提取兩個子任務的labels
    # 回歸: relatedness_score
    labels1 = [d['relatedness_score'] for d in batch]
    # 分類: entailment_judgment
    labels2 = [d['entailment_judgment'] for d in batch]

    # 標籤轉換為 PyTorch Tensors
    # 回歸 FloatTensor
    labels1_tensor = torch.tensor(labels1, dtype=torch.float)
    # 分類任務 LongTensor (用於計算 CrossEntropyLoss)
    labels2_tensor = torch.tensor(labels2, dtype=torch.long)

    # 5. 回傳一個字典，包含模型輸入和兩個任務的標籤
    return {
        "input_txt": inputs,       # 字典，包含 'input_ids', 'token_type_ids', 'attention_mask'
        "labels1": labels1_tensor, # 回歸任務
        "labels2": labels2_tensor  # 分類任務
    }


ds_train = SemevalDataset(split="train")
ds_validation = SemevalDataset(split="validation")
ds_test = SemevalDataset(split="test")


# 定義 DataLoader
dl_train = DataLoader(
    ds_train,
    batch_size=train_batch_size,
    shuffle=True,  # 打亂 
    collate_fn=collate_fn  
    )

dl_validation = DataLoader(
    ds_validation,
    batch_size=validation_batch_size,
    shuffle=False, # 不需要打亂
    collate_fn=collate_fn)

dl_test = DataLoader(
    ds_test,
    batch_size=1, 
    shuffle=False,
    collate_fn=collate_fn)


In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaModel 

# 模型
class MultiLabelModel(torch.nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # 載入 "roberta-base" 預訓練模型
        self.bert = RobertaModel.from_pretrained( 
            "roberta-base", 
            cache_dir="./cache_roberta/"
        )

        # 取得 RoBERTa 模型的輸出維度 (768)
        bert_hidden_size = self.bert.config.hidden_size

        # 加入 Dropout
        self.dropout = torch.nn.Dropout(0.1)

        # 定義回歸的線性層
        self.regressor = torch.nn.Linear(bert_hidden_size, 1)

        # 3. 定義分類的線性層
        self.classifier = torch.nn.Linear(bert_hidden_size, 3)

    def forward(self, input_txt):
        # 1. 將 input_txt 傳遞給 RoBERTa 模型

        input_dict = {
            'input_ids': input_txt['input_ids'],
            'attention_mask': input_txt['attention_mask']
        }

        outputs = self.bert(**input_dict)

        # 2. 取得 RoBERTa 的 pooler_output
        pooler_output = outputs.pooler_output

        dropped_output = self.dropout(pooler_output)

        # 3. 將 pooler_output 分別傳遞給兩個任務的頭
        output1 = self.regressor(dropped_output).squeeze(-1)
        output2 = self.classifier(dropped_output)

        # 4. 回傳輸出
        return output1, output2

print("RoBERTa-base Model Class (MultiLabelModel) defined successfully.")

RoBERTa-base Model Class (MultiLabelModel) defined successfully.


In [ ]:
# TODO3: Define your optimizer and loss function

model = MultiLabelModel().to(device)
# TODO3-1: Define your Optimizer
# 使用 AdamW 優化器，傳入模型所有可訓練的參數 (model.parameters()) 和學習率 (lr)
optimizer = AdamW(model.parameters(), lr=lr,weight_decay=0.01)

# TODO3-2: Define your loss functions (you should have two)
# 回歸任務的損失函數：Mean Squared Error Loss
loss_fn_1 = torch.nn.MSELoss()

# 分類的損失函數：Cross-Entropy Loss(會自動處理 logits(模型的原始輸出)和整數標籤)
loss_fn_2 = torch.nn.CrossEntropyLoss()

# scoring functions
# 載入 Hugging Face evaluate 的評分函式
psr = load("pearsonr")
acc = load("accuracy")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 用來儲存最佳分數
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train() # 設定為訓練模式

    total_train_loss = 0.0 # 計算平均訓練損失

    for batch in pbar:
        # 1. 資料移至device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 2. 清除梯度
        optimizer.zero_grad()

        # 3. 前向傳播
        outputs1, outputs2 = model(input_txt=input_txt)

        # 4. 計算損失
        loss1 = loss_fn_1(outputs1, labels1) # 回歸任務 (MSE)
        loss2 = loss_fn_2(outputs2, labels2) # 分類任務 (CrossEntropy)

        # 總損失
        loss = loss1 + loss2

        # 5. 反向傳播
        loss.backward()

        # 6. 模型優化
        optimizer.step()

        # 更新顯示
        pbar.set_postfix(loss=loss.item())
        total_train_loss += loss.item()

    print(f"Epoch {ep+1} Average Train Loss: {total_train_loss / len(dl_train):.4f}")

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval() # 設定為評估模式
    # TODO5: Write the evaluation loop
    # Write your code here
    # Evaluate your model
    with torch.no_grad():
        for batch in pbar:
            # 1. 將資料移至 device
            input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
            labels1 = batch['labels1'].to(device)
            labels2 = batch['labels2'].to(device)

            # 2. 執行模型 
            outputs1, outputs2 = model(input_txt=input_txt)

            # 3. 準備評分
            predictions2 = torch.argmax(outputs2, dim=1)

            # 結果加入評分器
            psr.add_batch(predictions=outputs1, references=labels1)
            acc.add_batch(predictions=predictions2, references=labels2)

    # 在迴圈結束，計算總體分數
    # Output all the evaluation scores (PearsonCorr, Accuracy)
    pearson_corr = psr.compute()['pearsonr']
    accuracy = acc.compute()['accuracy']
    if pearson_corr + accuracy > best_score:
        best = pearson_corr + accuracy
        torch.save(model.state_dict(), f'/content/drive/MyDrive/saved_models/best_RobertaModel.ckpt')


  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 1 Average Train Loss: 1.5557


  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 20 Average Train Loss: 0.1042


  0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
# TODO6: 測試迴圈

# 載入模型權重
print("Loading best model from best_RobertaModel.ckpt")
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"best_RobertaModel.ckpt", weights_only=True))

# 設定 model.eval()
pbar = tqdm(dl_test, desc="Test")
model.eval() # 將模型設定為評估模式

# 測試迴圈 
print("Starting evaluation on test set...")
with torch.no_grad():
    for batch in pbar:
        # 將資料移至 device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 執行模型 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 評分
        # outputs2 是 (batch_size, 3) 的 logits，我們需要 argmax 取得預測的類別
        predictions2 = torch.argmax(outputs2, dim=1)

        # 結果加入評分器
        psr.add_batch(predictions=outputs1, references=labels1)
        acc.add_batch(predictions=predictions2, references=labels2)

# 迴圈結束後，計算最終的測試集分數
test_pearson_corr = psr.compute()['pearsonr']
test_accuracy = acc.compute()['accuracy']

# 結果
print("--- Test Set Results ---")
print(f"Test Pearson Correlation: {test_pearson_corr:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("--------------------------")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading best model from /content/drive/MyDrive/saved_models/best_RobertaModel.ckpt


Test:   0%|          | 0/4927 [00:00<?, ?it/s]

Starting evaluation on test set...
--- Test Set Results ---
Test Pearson Correlation: 0.9063
Test Accuracy: 0.8947
--------------------------


# GPT-2 Model

In [ ]:
from transformers import GPT2Tokenizer 
# GPT-2
print("Loading GPT-2 Tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained(
    "gpt2", # 
    cache_dir="./cache_gpt2/"
)


# GPT-2 預設沒有 pad_token，設為 eos_token
tokenizer.pad_token = tokenizer.eos_token
print("GPT-2 Tokenizer loaded and pad_token set to eos_token.")

Loading GPT-2 Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

GPT-2 Tokenizer loaded and pad_token set to eos_token.


In [ ]:
# PyTorch Dataset類別，用來處理SemEval 2014 Task 1資料集
class SemevalDataset(Dataset):
    # 初始化函式，參數 split 用來指定要載入 'train', 'validation', 或是 'test' 資料集
    def __init__(self, split="train") -> None:
        super().__init__()
        # 確保傳入的 split 參數是合法的
        assert split in ["train", "validation", "test"]
        # 使用 Hugging Face 的 datasets 函式庫載入 "sem_eval_2014_task_1" 資料集
        # trust_remote_code=True 是因為這個資料集需要執行遠端程式碼來載入
        # .to_list() 載入的資料轉為Python 列表
        self.data = load_dataset(
            "sem_eval_2014_task_1", split=split, trust_remote_code=True, cache_dir="./cache/"
        ).to_list()

    # 定義如何根據索引取得單筆資料
    def __getitem__(self, index):
        # 根據索引從 self.data中取出原始資料
        d = self.data[index]
        # 針對 "premise" (前提) 和 "hypothesis" (假設) 這兩個欄位，執行標點符號的替換
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d
    # 總資料筆數
    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Generating train split:   0%|          | 0/4500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4927 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [ ]:
# 超參數
lr = 2e-5
epochs = 20
train_batch_size = 16
validation_batch_size = 16

In [ ]:
def collate_fn(batch):
    #將 'premise' 和 'hypothesis' 欄位分別抽出來，放到兩個列表
    premises = [d['premise'] for d in batch]
    hypotheses = [d['hypothesis'] for d in batch]

    # 使用 tokenizer 將文本對(premise,hypothesis)轉換為模型輸入格式
    # BERT 會自動處理成 [CLS] premise [SEP] hypothesis [SEP]
    inputs = tokenizer(
        premises,
        hypotheses,
        padding=True,
        truncation=True,
        return_tensors="pt",
        padding_side='left'  # <--- 關鍵：GPT-2 必須用 left-padding
    )

    # 提取兩個子任務的labels
    # 回歸
    labels1 = [d['relatedness_score'] for d in batch]
    # 分類
    labels2 = [d['entailment_judgment'] for d in batch]

    # 將標籤轉換為 PyTorch Tensors
    # 回歸任 FloatTensor
    labels1_tensor = torch.tensor(labels1, dtype=torch.float)
    # 分類任務 LongTensor (用於計算 CrossEntropyLoss)
    labels2_tensor = torch.tensor(labels2, dtype=torch.long)

    # 回傳字典，含模型輸入和兩個任務的標籤
    return {
        "input_txt": inputs,       # 字典，包含 'input_ids', 'token_type_ids', 'attention_mask'
        "labels1": labels1_tensor, # 回歸任務的標籤
        "labels2": labels2_tensor  # 分類任務的標籤
    }

# 定義 DataLoader
ds_train = SemevalDataset(split="train")
ds_validation = SemevalDataset(split="validation")
ds_test = SemevalDataset(split="test")

dl_train = DataLoader(
    ds_train,
    batch_size=train_batch_size,
    shuffle=True,  # 訓練資料集需要打亂 (shuffle)
    collate_fn=collate_fn  # 指定我們剛剛定義的 collate_fn
    )

dl_validation = DataLoader(
    ds_validation,
    batch_size=validation_batch_size,
    shuffle=False, # 驗證和測試資料集不需要打亂
    collate_fn=collate_fn)

dl_test = DataLoader(
    ds_test,
    batch_size=1, # 測試集的 batch size (作業沒給，先用 validation 的)
    shuffle=False,
    collate_fn=collate_fn)


In [ ]:

import torch
import torch.nn as nn
from transformers import GPT2Model # <--- 變更 (1)

# 模型 GPT-2 Version
class MultiLabelModel(torch.nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # 載入 "gpt2" 預訓練模型
        self.bert = GPT2Model.from_pretrained( # <--- 變更 (2)
            "gpt2", # <--- 變更 (3)
            cache_dir="./cache_gpt2/"
        )

        # 取得 GPT-2 模型的輸出維度 ( 768)
        bert_hidden_size = self.bert.config.hidden_size

        # 加入 Dropout
        self.dropout = torch.nn.Dropout(0.1)

        # 定義兩個任務的頭
        self.regressor = torch.nn.Linear(bert_hidden_size, 1)
        self.classifier = torch.nn.Linear(bert_hidden_size, 3)

    def forward(self, input_txt):
        # 準備輸入
        # GPT-2 不使用 'token_type_ids'
        input_ids = input_txt['input_ids']
        attention_mask = input_txt['attention_mask']
        batch_size = input_ids.shape[0]

        # 將輸入傳遞給 GPT-2
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # --- 取得最後一個 token 的特徵 ---
        # 取得最後一層的隱藏狀態
        # shape: (batch_size, sequence_length, hidden_size)
        last_hidden_state = outputs.last_hidden_state

        # 找出每個序列的長度 
        # attention_mask.sum(dim=1) 會給我們每個序列的實際長度
        # 長度 10 的序列，最後一個 token 的 index 是 9
        sequence_lengths = attention_mask.sum(dim=1)
        last_token_indices = sequence_lengths - 1

        # 提取每個序列的「最後一個」 token 的特徵
        # 用 [torch.arange(batch_size), last_token_indices] 進行 fancy indexing
        features = last_hidden_state[torch.arange(batch_size), last_token_indices]

        # Dropout
        dropped_features = self.dropout(features)

        # 特徵傳遞給兩個任務的頭
        output1 = self.regressor(dropped_features).squeeze(-1)
        output2 = self.classifier(dropped_features)

        return output1, output2

print("GPT-2 Model Class (MultiLabelModel) defined successfully.")

GPT-2 Model Class (MultiLabelModel) defined successfully.


In [ ]:


model = MultiLabelModel().to(device)
# 使用 AdamW 優化器，傳入模型所有可訓練的參數 (model.parameters()) 和學習率 (lr)
optimizer = AdamW(model.parameters(), lr=lr,weight_decay=0.01)

# 回歸任務的損失函數：Mean Squared Error Loss
loss_fn_1 = torch.nn.MSELoss()

# 分類的損失函數：Cross-Entropy Loss(會自動處理 logits(模型的原始輸出)和整數標籤)
loss_fn_2 = torch.nn.CrossEntropyLoss()

# 載入 Hugging Face evaluate 的評分函式
psr = load("pearsonr")
acc = load("accuracy")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
# 用來儲存最佳分數
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train() # 訓練模式

    total_train_loss = 0.0 # 計算平均訓練損失

    for batch in pbar:
        # 1. 資料移至device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 2. 清除梯度
        optimizer.zero_grad()

        # 3. 前向傳播 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 4. 計算損失
        loss1 = loss_fn_1(outputs1, labels1) # 回歸
        loss2 = loss_fn_2(outputs2, labels2) # 分類

        # 總損失
        loss = loss1 + loss2

        # 5. 反向傳播
        loss.backward()

        # 6. 模型優化
        optimizer.step()

        # 更新顯示
        pbar.set_postfix(loss=loss.item())
        total_train_loss += loss.item()

    print(f"Epoch {ep+1} Average Train Loss: {total_train_loss / len(dl_train):.4f}")

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval() # 設定為評估模式
    # TODO5: Write the evaluation loop
    # Write your code here
    # Evaluate your model
    with torch.no_grad():
        for batch in pbar:
            # 1. 將資料移至 device
            input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
            labels1 = batch['labels1'].to(device)
            labels2 = batch['labels2'].to(device)

            # 2. 執行模型 
            outputs1, outputs2 = model(input_txt=input_txt)

            # 3. 評分
            predictions2 = torch.argmax(outputs2, dim=1)

            # 4. 這批次的結果加入評分器
            psr.add_batch(predictions=outputs1, references=labels1)
            acc.add_batch(predictions=predictions2, references=labels2)

    # 5. 計算總體分數
    pearson_corr = psr.compute()['pearsonr']
    accuracy = acc.compute()['accuracy']
    if pearson_corr + accuracy > best_score:
        best = pearson_corr + accuracy
        torch.save(model.state_dict(), f'/content/drive/MyDrive/saved_models/best_gpt2_0.ckpt')


  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 1 Average Train Loss: 2.7949


  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 20 Average Train Loss: 1.7380


  0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
# 測試迴圈

# 載入模型權重
print("Loading best model from best_gpt2_0.ckpt")
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"best_gpt2_0.ckpt", weights_only=True))

# 設定 pbar 和 model.eval()
pbar = tqdm(dl_test, desc="Test")
model.eval() # 將模型設定為評估模式

# 執行測試迴圈 
print("Starting evaluation on test set...")
with torch.no_grad():
    for batch in pbar:
        #將資料移至 device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)
        # 執行模型 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 評分
        predictions2 = torch.argmax(outputs2, dim=1)

        # 將這一批次的結果加入評分器
        psr.add_batch(predictions=outputs1, references=labels1)
        acc.add_batch(predictions=predictions2, references=labels2)

# 在迴圈結束後，計算最終的測試集分數
test_pearson_corr = psr.compute()['pearsonr']
test_accuracy = acc.compute()['accuracy']

# 結果
print("--- Test Set Results ---")
print(f"Test Pearson Correlation: {test_pearson_corr:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("--------------------------")

Loading best model from /content/drive/MyDrive/saved_models/best_gpt2_0.ckpt


Test:   0%|          | 0/4927 [00:00<?, ?it/s]

Starting evaluation on test set...
--- Test Set Results ---
Test Pearson Correlation: 0.7074
Test Accuracy: 0.7668
--------------------------


# 錯誤分析

In [ ]:
import torch
import pandas as pd
from tqdm.notebook import tqdm

print("--- 開始執行獨立的錯誤分析 ---")

# 載入最佳模型
print("Loading best model from best_model_4.ckpt")
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"best_model_4.ckpt", weights_only=True))
model.eval() # 設定為評估模式

# 準備儲存預測結果的列表
all_preds_1 = [] # 儲存 回歸預測 
all_preds_2 = [] # 儲存 分類預測 

# 執行預測迴圈 
print("Running predictions on validation set (dl_validation)...")
with torch.no_grad():
    for batch in tqdm(dl_validation):
        # 將資料移至 device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
       
        # 執行模型 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 處理預測結果
        predictions2 = torch.argmax(outputs2, dim=1)

        # 儲存結果 
        all_preds_1.extend(outputs1.cpu().numpy())
        all_preds_2.extend(predictions2.cpu().numpy())

print("預測完成！")

# --- 建立 DataFrame 並進行分析 ---
print("Generating error analysis report...")

# 載入原始的驗證集資料
original_data = ds_validation.data
df = pd.DataFrame(original_data)

# 加入我們儲存的最佳預測結果
df['pred_score'] = all_preds_1
df['pred_judgment'] = all_preds_2

# 找出錯誤範例

# --- 1 分析子任務1 (回歸) ---
# 計算預測分數和真實分數的差距
df['score_diff'] = (df['relatedness_score'] - df['pred_score']).abs()

# 找出差距最大的前 10 筆
print("\n--- 錯誤分析: 回歸任務 (差距最大) ---")
pd.set_option('display.max_colwidth', 200) # 避免文字被截斷
print(df.nlargest(10, 'score_diff')[['premise', 'hypothesis', 'relatedness_score', 'pred_score', 'score_diff']])


# --- 2 分析子任務2 (分類) ---
# 找出預測錯誤的資料
df_wrong_cls = df[df['entailment_judgment'] != df['pred_judgment']]

print(f"\n--- 錯誤分析: 分類任務 (預測錯誤) ---")
print(f"總共錯誤 {len(df_wrong_cls)} 筆 / 總共 {len(df)} 筆 (Accuracy: {1 - len(df_wrong_cls)/len(df):.4f})")
print(df_wrong_cls.head(10)[['premise', 'hypothesis', 'entailment_judgment', 'pred_judgment']])

--- 開始執行獨立的錯誤分析 ---
Loading best model from /content/drive/MyDrive/saved_models/best_model_4.ckpt
Running predictions on validation set (dl_validation)...


  0%|          | 0/32 [00:00<?, ?it/s]

預測完成！
Generating error analysis report...

--- 錯誤分析: 回歸任務 (差距最大) ---
                                                                                               premise  \
320  A white dog is wearing a Christmas reindeer headband and is playing with a brown dog in the grass   
366                                          An American footballer is wearing the red and white strip   
78                                                               A snake is being fed a mouse by a man   
396                  A woman dressed in elegant clothing is inside a crowd of people and is looking up   
96                                                                     The man is denying an interview   
302                 A young girl in a blue shirt is walking on the sidewalk and holding up a pink sign   
191                                                                     It is raining on a walking man   
488                                        Two small children are playing with a to

# 類別不平衡 vs 任務衝突診斷

In [ ]:

model = MultiLabelModel().to(device)
# 使用 AdamW 優化器，傳入模型所有可訓練的參數 (model.parameters()) 和學習率 (lr)
optimizer = AdamW(model.parameters(), lr=lr,weight_decay=0.01)

# 定義 Loss 1 
loss_fn_1 = nn.MSELoss()

# 定義 Loss 2
loss_fn_2 = nn.CrossEntropyLoss(weight=class_weights_tensor) 

# 載入 Hugging Face evaluate 的評分函式
psr = load("pearsonr")
acc = load("accuracy")

In [ ]:
# 用來儲存最佳分數
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train() # 設定為訓練模式

    total_train_loss = 0.0 # 計算平均訓練損失

    for batch in pbar:
        # 1. 資料移至device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 2. 清除梯度
        optimizer.zero_grad()

        # 3. 前向傳播 
        outputs1, outputs2 = model(input_txt=input_txt)

        # 4. 計算損失
        loss1 = loss_fn_1(outputs1, labels1) # 回歸任務 
        loss2 = loss_fn_2(outputs2, labels2) # 分類任務 

        # 總損失
        # loss = loss1 + loss2
        loss = loss1 + loss2


        # 5. 反向傳播
        loss.backward()

        # 6. 模型優化
        optimizer.step()

        # 更新 pbar 上的顯示
        pbar.set_postfix(loss=loss.item())
        total_train_loss += loss.item()

    print(f"Epoch {ep+1} Average Train Loss: {total_train_loss / len(dl_train):.4f}")

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval() # 設定為評估模式
    
    
    # 評估
    with torch.no_grad():
        for batch in pbar:
            # 1. 將資料移至 device
            input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
            labels1 = batch['labels1'].to(device)
            labels2 = batch['labels2'].to(device)

            # 2. 執行模型 (Forward pass)
            outputs1, outputs2 = model(input_txt=input_txt)

            # 3. 準備評分
            # outputs2 是 (batch_size, 3) 的 logits，我們需要 argmax 取得預測的類別 (0, 1, or 2)
            predictions2 = torch.argmax(outputs2, dim=1)

            # 4. 將這一批次的結果加入評分器
            psr.add_batch(predictions=outputs1, references=labels1)
            acc.add_batch(predictions=predictions2, references=labels2)

    # 5. 計算總體分數
    # Output all the evaluation scores (PearsonCorr, Accuracy)
    pearson_corr = psr.compute()['pearsonr']
    accuracy = acc.compute()['accuracy']
    # print(f"F1 Score: {f1.compute()}")
    # print(f"Epoch {ep+1} Validation Pearson: {pearson_corr:.4f}, Accuracy: {accuracy:.4f}")
    if pearson_corr + accuracy > best_score:
        best = pearson_corr + accuracy
        torch.save(model.state_dict(), f'/content/drive/MyDrive/saved_models/best_model_tune.ckpt')


  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 1 Average Train Loss: 1.4519


  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch 20 Average Train Loss: 0.0618


  0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
# 測試迴圈

# 模型權重
print("Loading best model from best_model_tune.ckpt")
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"best_model_tune.ckpt", weights_only=True))

# 設定 pbar 和 model.eval()
pbar = tqdm(dl_test, desc="Test")
model.eval() # 將模型設定為評估模式

# 執行測試迴圈 (不需要計算梯度)
print("Starting evaluation on test set...")
with torch.no_grad():
    for batch in pbar:
        # 將資料移至 device
        input_txt = {k: v.to(device) for k, v in batch['input_txt'].items()}
        labels1 = batch['labels1'].to(device)
        labels2 = batch['labels2'].to(device)

        # 執行模型 (Forward pass)
        outputs1, outputs2 = model(input_txt=input_txt)

        # 準備評分
        # outputs2 是 (batch_size, 3) 的 logits，我們需要 argmax 取得預測的類別
        predictions2 = torch.argmax(outputs2, dim=1)

        # 將這一批次的結果加入評分器
        psr.add_batch(predictions=outputs1, references=labels1)
        acc.add_batch(predictions=predictions2, references=labels2)

# 在迴圈結束後，計算最終的測試集分數
test_pearson_corr = psr.compute()['pearsonr']
test_accuracy = acc.compute()['accuracy']

# 結果
print("--- Test Set Results ---")
print(f"Test Pearson Correlation: {test_pearson_corr:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("--------------------------")

Loading best model from /content/drive/MyDrive/saved_models/best_model_tune.ckpt


Test:   0%|          | 0/4927 [00:00<?, ?it/s]

Starting evaluation on test set...
--- Test Set Results ---
Test Pearson Correlation: 0.8778
Test Accuracy: 0.8502
--------------------------


In [1]:
# 類別權重計算

In [ ]:
from collections import Counter
import torch
import torch.nn as nn
import numpy as np

print("--- Step 1: Calculating Class Weights for Task 2 ---")

# 1. 計算每個類別的樣本數
label_counts = Counter([d['entailment_judgment'] for d in ds_train.data])

# 排序以確保順序是 [Count(0), Count(1), Count(2)]
counts = [label_counts[i] for i in range(3)]
print(f"Label Counts: {label_counts}")
print(f"Counts (sorted): {counts}")

# 2. 計算逆樣本頻率作為權重
# 權重 = 總樣本數 / (類別數 * 該類別樣本數)
total_samples = sum(counts)
num_classes = len(counts)

class_weights = []
for count in counts:
    weight = total_samples / (num_classes * count)
    class_weights.append(weight)

# 3. 將權重normalize，讓最小的權重為 1.0
min_weight = min(class_weights)
class_weights = [w / min_weight for w in class_weights]

# 4. 轉換為 PyTorch Tensor 並移至 device
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Calculated Weights: {class_weights}")
print(f"Final Weights Tensor (on {device}): {class_weights_tensor}")

--- Step 1: Calculating Class Weights for Task 2 ---
Label Counts: Counter({0: 2536, 1: 1299, 2: 665})
Counts (sorted): [2536, 1299, 665]
Calculated Weights: [1.0, 1.9522709776751346, 3.813533834586466]
Final Weights Tensor (on cuda): tensor([1.0000, 1.9523, 3.8135], device='cuda:0')
